# Clase 14 — Comunicar resultados: reporte, figuras y exposición
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 12 de noviembre**

Tres ejercicios sobre el mini-caso de **pedidos de un servicio de reparto a domicilio** que ya conocen del Lab 11 (la pregunta era **"¿este pedido llegará con retraso?"**). El modelo ya está hecho y evaluado: hoy el trabajo es **contar el resultado** a quien lo va a leer. Al final, la parte de proyecto: el resumen ejecutivo y el *pitch* de 3 minutos de su propio trabajo.

**Datos sintéticos generados para el lab: ninguna cifra describe a una empresa, un comercio o un reparto real.**

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | Reescribir un resumen ejecutivo | la conclusión primero, cifras traducidas para no técnicos, una frase técnica con cifra + incertidumbre + comparación + origen | 20 |
| 2 | Arreglar tres figuras engañosas | eje truncado, demasiadas series, título genérico sin incertidumbre ni baseline | 25 |
| 3 | Tabla de resultados lista para el informe | media ± desviación, orden, mejor resaltado, empate práctico, exportar a `reports/`; (extra) página de resumen y HTML opcional | 25 |

Reglas del notebook:
- El archivo de práctica **se genera solo** en `data_practica/`: no hay que descargar nada y el lab no usa internet.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓). Las soluciones se revisan en clase.
- `plotly` es **opcional**: solo lo usa la parte B del extra 3.3. Sin él, el lab funciona completo.
- Referencia técnica: `lab14-guia-comunicar-resultados.md`.

In [ ]:
# Setup — ejecutar primero
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.container import ErrorbarContainer

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

try:
    import sklearn
except ImportError:
    print("scikit-learn no está instalado: ejecuten  pip install scikit-learn  y reinicien el kernel")
    raise ImportError("Este lab necesita scikit-learn: pip install scikit-learn") from None

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    import plotly.express as px
    HAY_PLOTLY = True
except ImportError:
    HAY_PLOTLY = False
    print("plotly no está instalado: la parte B del extra 3.3 se omite (no es un error).")

SEED = 2026
CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)
FIGS = Path("figs")
REPORTS = Path("reports")


def guardar_figura(fig, nombre):
    """Guarda una figura en figs/ (la crea si no existe) con la resolución del informe."""
    FIGS.mkdir(exist_ok=True)
    ruta = FIGS / nombre
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)


def fmt(x, d=3):
    """Número con coma decimal: fmt(0.6161) -> '0,616'."""
    return f"{x:.{d}f}".replace(".", ",")


def df_a_markdown(tabla, ruta, nota=""):
    """Escribe una tabla de pandas como tabla Markdown (sin dependencias extra) y una nota al pie."""
    ruta = Path(ruta)
    ruta.parent.mkdir(exist_ok=True)
    nombre_indice = tabla.index.name or ""
    columnas = [nombre_indice] + [str(c) for c in tabla.columns]
    lineas = ["| " + " | ".join(columnas) + " |", "|" + "---|" * len(columnas)]
    for idx, fila in tabla.iterrows():
        lineas.append("| " + " | ".join([str(idx)] + [str(v) for v in fila]) + " |")
    texto = "\n".join(lineas) + ("\n\n" + nota if nota else "") + "\n"
    ruta.write_text(texto, encoding="utf-8")
    print("Guardado:", ruta)
    return texto


def es_gris(color):
    """True si el color es un gris (r ≈ g ≈ b)."""
    r, g, b = to_rgb(color)
    return max(r, g, b) - min(r, g, b) < 0.08


TITULOS_GENERICOS = {"", "resultados", "resultado", "gráfico", "grafico", "figura", "comparación", "comparacion",
                     "comparación de modelos", "comparacion de modelos", "tasa de retraso", "retrasos",
                     "tasa de retraso por canal", "tasa de retraso por zona y franja horaria", "métricas", "metricas"}


def titulo_es_mensaje(ax, minimo=25):
    """Un título-mensaje dice la conclusión: no es genérico y tiene al menos `minimo` caracteres."""
    t = ax.get_title().strip()
    return t.lower() not in TITULOS_GENERICOS and len(t) >= minimo


def textos_de(fig):
    """Todos los textos de una figura (títulos, ejes, anotaciones y notas al pie)."""
    out = [t.get_text() for t in fig.texts]
    for ax in fig.axes:
        out += [ax.get_title(), ax.get_xlabel(), ax.get_ylabel()] + [t.get_text() for t in ax.texts]
    return " ".join(out)


def tiene_nota_n(fig):
    return re.search(r"\bn\s*=", textos_de(fig)) is not None


def ejecutar_checks(checks):
    """checks: lista de (nombre, funcion_booleana_sin_argumentos).
    Atrapa NameError para que "Restart & Run All" con TU TURNO vacíos no se caiga."""
    faltan = 0
    for nombre, chk in checks:
        try:
            ok = bool(chk())
        except NameError as e:
            var = str(e).split("'")[1]
            print(f"✗ {nombre} (todavía no definen `{var}`)")
            faltan += 1
            continue
        except Exception as e:
            print(f"✗ {nombre} (todavía no está listo: {type(e).__name__})")
            faltan += 1
            continue
        print(("✓" if ok else "✗"), nombre)
        faltan += not ok
    print("Todo OK" if faltan == 0 else f"Faltan {faltan}")
    return faltan == 0


print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· scikit-learn:", sklearn.__version__,
      "· plotly:", "sí" if HAY_PLOTLY else "no (opcional)")
print("Carpeta de trabajo:", CARPETA.resolve())

In [ ]:
# Genera el archivo de práctica (no modificar esta celda: es la misma del Lab 11)
rng = np.random.default_rng(2026)
N = 6000
N_COMERCIOS = 500

ZONAS = ["Centro", "Norte", "Sur", "Costa", "Cerro", "Periferia"]
EFECTO_ZONA = {"Centro": 0.0, "Norte": 0.1, "Sur": 0.15, "Costa": 0.35, "Cerro": 0.6, "Periferia": 0.5}
VEHICULOS = ["moto", "bici", "auto"]
CANALES = ["app", "web", "telefono"]

id_comercio = rng.integers(0, N_COMERCIOS, N)
efecto_comercio = rng.normal(0, 0.45, N_COMERCIOS)

zona = rng.choice(ZONAS, size=N, p=[0.28, 0.17, 0.17, 0.10, 0.10, 0.18])
distancia = np.clip(rng.gamma(2.2, 1.8, N) + 0.5, 0.5, 18).round(1)
n_items = np.clip(1 + rng.poisson(2.5, N), 1, 15)
_pesos_hora = np.array([3, 5, 9, 10, 5, 3, 3, 4, 7, 10, 10, 6, 3, 2])
hora = rng.choice(np.arange(10, 24), size=N, p=_pesos_hora / _pesos_hora.sum())
dia_semana = rng.integers(0, 7, N)
lluvia = (rng.random(N) < 0.2).astype(int)
punta = np.isin(hora, [13, 14, 20, 21]).astype(int)
trafico = np.clip(rng.normal(4.5 + 1.6 * punta + 0.9 * lluvia, 1.8, N), 0, 10).round(1)
prep = np.clip(rng.normal(12 + 1.3 * n_items, 4, N), 4, 60).round(0).astype(int)
antig = np.clip(rng.exponential(14, N), 0, 60).round(0).astype(int)
vehiculo = rng.choice(VEHICULOS, size=N, p=[0.60, 0.25, 0.15])
canal = rng.choice(CANALES, size=N, p=[0.70, 0.22, 0.08])

z = (-4.55 + 0.24 * distancia + 0.40 * lluvia + 0.50 * punta + 0.20 * (trafico - 4.5)
     + 0.07 * (prep - 15) - 0.02 * antig
     + 1.5 * np.array([EFECTO_ZONA[k] for k in zona])
     + 0.10 * distancia * (vehiculo == "bici") + 0.70 * lluvia * punta
     + 0.50 * (dia_semana >= 5) + efecto_comercio[id_comercio])
p = 1 / (1 + np.exp(-z))
retraso = (rng.random(N) < p).astype(int)

pedidos = pd.DataFrame({
    "id_pedido": [f"P{i:05d}" for i in range(1, N + 1)],
    "id_comercio": [f"C{c:03d}" for c in id_comercio],
    "zona": zona, "canal": canal, "tipo_vehiculo": vehiculo,
    "distancia_km": distancia, "n_items": n_items, "hora_pedido": hora, "dia_semana": dia_semana,
    "lluvia": lluvia, "trafico_idx": trafico, "tiempo_prep_min": prep,
    "antiguedad_repartidor_meses": antig, "retraso": retraso,
})
pedidos.to_csv(CARPETA / "pedidos_entregas.csv", index=False, encoding="utf-8")
print("pedidos_entregas.csv:", pedidos.shape)
print(f"Escrito en {CARPETA.resolve()} · tasa de retraso: {pedidos.retraso.mean():.1%}")

### Setup de los resultados: el modelado ya está hecho

Esta celda **rehace en 2 segundos** lo que se decidió en el Lab 11, para tener cifras de verdad que comunicar: test separado al inicio (20 %), los **mismos 5 folds** para todas las opciones, el costo declarado y el **umbral por costo** 0,09 (`C_FP / (C_FP + C_FN)`). Deja dos objetos:

- `resultados_cv`: una fila por opción y fold, con `costo_por_pedido`, `sensibilidad` y `pr_auc` (dos políticas triviales y tres modelos).
- `resultado_test`: el resultado **final** del modelo elegido (regresión logística) en el test, evaluado **una sola vez**.

> *No avisarle al cliente de un retraso (falso negativo) cuesta 10 unidades (reclamo y compensación); avisarle de un retraso que no ocurre (falso positivo) cuesta 1 unidad (un mensaje de más).*

No hay nada que modelar hoy. Ver `lab14-guia-comunicar-resultados.md`, §0.

In [ ]:
# Resultados del modelado (no modificar esta celda)
df = pd.read_csv(CARPETA / "pedidos_entregas.csv", dtype={"id_pedido": str, "id_comercio": str})
X = pd.get_dummies(df.drop(columns=["id_pedido", "id_comercio", "retraso"]),
                   columns=["zona", "canal", "tipo_vehiculo"], dtype=int)
y = df["retraso"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
C_FN, C_FP = 10, 1
UMBRAL = C_FP / (C_FP + C_FN)   # 0,09


def costo_por_pedido(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return float((C_FN * fn + C_FP * fp) / len(y_real))


def crear_modelos():
    return {
        "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Árbol de decisión": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED),
        "Bosque aleatorio": RandomForestClassifier(n_estimators=200, min_samples_leaf=10,
                                                   random_state=SEED, n_jobs=-1),
    }


_filas = []
for f, (a, b) in enumerate(CV.split(X_tr, y_tr), start=1):
    ya, yb = y_tr.iloc[a], y_tr.iloc[b]
    _filas.append({"modelo": "Nunca avisar", "fold": f, "costo_por_pedido": costo_por_pedido(yb, np.zeros(len(yb), int)),
                   "sensibilidad": 0.0, "pr_auc": yb.mean()})
    _filas.append({"modelo": "Avisar siempre", "fold": f, "costo_por_pedido": costo_por_pedido(yb, np.ones(len(yb), int)),
                   "sensibilidad": 1.0, "pr_auc": yb.mean()})
    for nombre, modelo in crear_modelos().items():
        modelo.fit(X_tr.iloc[a], ya)
        proba = modelo.predict_proba(X_tr.iloc[b])[:, 1]
        pred = (proba >= UMBRAL).astype(int)
        _filas.append({"modelo": nombre, "fold": f, "costo_por_pedido": costo_por_pedido(yb, pred),
                       "sensibilidad": recall_score(yb, pred), "pr_auc": average_precision_score(yb, proba)})
resultados_cv = pd.DataFrame(_filas)

_elegido = crear_modelos()["Regresión logística"].fit(X_tr, y_tr)
_pred = (_elegido.predict_proba(X_te)[:, 1] >= UMBRAL).astype(int)
_tn, _fp, _fn, _tp = confusion_matrix(y_te, _pred, labels=[0, 1]).ravel()
resultado_test = {
    "modelo": "Regresión logística", "umbral": UMBRAL, "n": len(y_te),
    "costo_por_pedido": costo_por_pedido(y_te, _pred),
    "costo_siempre": costo_por_pedido(y_te, np.ones(len(y_te), int)),
    "costo_nunca": costo_por_pedido(y_te, np.zeros(len(y_te), int)),
    "sensibilidad": float(recall_score(y_te, _pred)),
    "retrasos": int(_tp + _fn), "avisados": int(_tp), "avisos": int(_tp + _fp), "avisos_de_mas": int(_fp),
}
print("resultados_cv:", resultados_cv.shape, "· opciones:", list(resultados_cv.modelo.unique()))
print("resultado_test:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in resultado_test.items()})

### El informe borrador (lo que hay que arreglar hoy)

Un equipo ficticio entregó este borrador. **Todo lo que dice es cierto**, pero está mal comunicado:

> **Resumen.** En este trabajo se utilizaron diversas técnicas de machine learning sobre los datos de pedidos de la empresa. Se probaron varios modelos, entre ellos Random Forest, árboles y regresión logística, y se ajustaron sus parámetros. Se obtuvieron buenos resultados, con un PR-AUC de 0,35. Se concluye que el modelo es muy bueno y puede ser útil para la empresa.

Y venía con tres figuras (Ejercicio 2) y una tabla copiada de la consola (Ejercicio 3).

---
## Ejercicio 1 — Reescribir un resumen ejecutivo (≈20 min)

El lector de un resumen ejecutivo tiene un minuto. Lo que necesita, en orden: **la pregunta**, **el hallazgo** (una cifra con su comparación), **qué significa**, **qué se recomienda** y **qué limita la conclusión**. El método va después, en el cuerpo del informe. Y para el lector técnico hay otra regla: **cifra + incertidumbre + comparación + dónde se calculó**.

Ver `lab14-guia-comunicar-resultados.md`, §1 y §2.

### 1.1 Primero, el error

In [ ]:
borrador = ("En este trabajo se utilizaron diversas técnicas de machine learning sobre los datos de pedidos de la "
            "empresa. Se probaron varios modelos, entre ellos Random Forest, árboles y regresión logística, y se "
            "ajustaron sus parámetros. Se obtuvieron buenos resultados, con un PR-AUC de 0,35. Se concluye que el "
            "modelo es muy bueno y puede ser útil para la empresa.")
print(borrador)
print()
print("Cifras en el resumen:", re.findall(r"\d+,\d+", borrador))
print("Pregunta: si usted es la gerente de operaciones, ¿qué decisión puede tomar con esto?")

*Una sola cifra, sin comparación, en una métrica que la gerente no conoce. ¿0,35 es bueno? ¿Frente a qué?*

### 1.2 ✏️ TU TURNO
Con `resultado_test` (el resultado final en el test) y `resultados_cv`:

1. `cifras` = diccionario con `costo_modelo` (costo por pedido del modelo en el test), `costo_siempre` (de la política "avisar siempre" en el test), `reduccion_vs_siempre` (fracción entre 0 y 1: cuánto baja el costo frente a "avisar siempre"), `avisados` (retrasos que el modelo avisó), `retrasos` (retrasos totales del test) y `avisos_de_mas` (avisos que resultaron innecesarios).
2. `resumen` = diccionario con las cinco claves `pregunta`, `hallazgo`, `significado`, `recomendacion` y `limitacion`, cada una con **una o dos frases para una gerente que no es técnica**:
   - en `pregunta`, `hallazgo`, `significado` y `recomendacion`, **sin jerga**: nada de PR-AUC, AUC, F1, folds, hiperparámetros ni nombres de algoritmos;
   - el `hallazgo` lleva **una cifra y su comparación** (por ejemplo, contra avisar a todos o contra no tener modelo), traducida a lenguaje cotidiano ("de cada 10…");
   - la `limitacion` es **concreta** para este caso, no "faltan datos".
3. `frase_tecnica` = **una** frase para el lector técnico con la regla completa: la cifra del modelo **con su desviación (±)** en validación cruzada, la **comparación** con la mejor política trivial y **dónde se calculó** (variable o celda).
4. En la celda de texto: léanle el `resumen` a la persona de al lado sin mostrarle la pantalla. ¿Pudo repetir el mensaje?
5. `conclusion_ej1` = `"a"`, `"b"` o `"c"`:
   - a) El borrador está casi bien: tiene una cifra (PR-AUC 0,35); basta con explicar mejor las técnicas que se usaron.
   - b) El resumen debe partir por el hallazgo, con una cifra comparada contra no usar el modelo y traducida para quien decide; el PR-AUC sin comparación no le dice nada a la gerente.
   - c) En un resumen ejecutivo no van cifras: solo conclusiones cualitativas, para no confundir.

In [ ]:
# ✏️ TU TURNO
cifras = {
    "costo_modelo": ...,
    "costo_siempre": ...,
    "reduccion_vs_siempre": ...,   # (costo_siempre - costo_modelo) / costo_siempre
    "avisados": ...,
    "retrasos": ...,
    "avisos_de_mas": ...,
}

resumen = {
    "pregunta": "",
    "hallazgo": "",
    "significado": "",
    "recomendacion": "",
    "limitacion": "",
}

# media y desviación del costo de cada opción en validación cruzada:
# resultados_cv.groupby("modelo")["costo_por_pedido"].agg(["mean", "std"])
frase_tecnica = ""

conclusion_ej1 = ""

**¿Pudo repetir el mensaje la persona de al lado? ¿Qué palabra tuvo que explicarle?** *(escriban aquí)*

### ✅ Verificación automática

In [ ]:
JERGA = ["pr-auc", "auc", "f1", "fold", "hiperparámetro", "hiperparametro", "random forest", "bosque aleatorio",
         "regresión logística", "regresion logistica", "árbol de decisión", "arbol de decision", "sklearn",
         "machine learning", "umbral", "sensibilidad", "precision", "precisión"]
COMPARACION = ["frente a", "comparado", "en vez de", "en lugar de", "que avisar", "sin modelo", "sin el sistema",
               "a todos", "siempre", "sin avisar", "que no avisar"]


def _sin_jerga():
    texto = " ".join(resumen[k] for k in ["pregunta", "hallazgo", "significado", "recomendacion"]).lower()
    return not any(j in texto for j in JERGA)


def verificar_ej1():
    r = resultado_test
    return ejecutar_checks([
        ("cifras: costo del modelo y de 'avisar siempre' en el test",
         lambda: abs(cifras["costo_modelo"] - r["costo_por_pedido"]) < 0.005
         and abs(cifras["costo_siempre"] - r["costo_siempre"]) < 0.005),
        ("cifras: reducción frente a 'avisar siempre' (≈ 34 %)",
         lambda: abs(cifras["reduccion_vs_siempre"] - (r["costo_siempre"] - r["costo_por_pedido"]) / r["costo_siempre"]) < 0.01),
        ("cifras: avisados, retrasos y avisos de más",
         lambda: (cifras["avisados"], cifras["retrasos"], cifras["avisos_de_mas"])
         == (r["avisados"], r["retrasos"], r["avisos_de_mas"])),
        ("resumen: las cinco partes, cada una con al menos 25 caracteres",
         lambda: set(resumen) == {"pregunta", "hallazgo", "significado", "recomendacion", "limitacion"}
         and all(len(str(v).strip()) >= 25 for v in resumen.values())),
        ("resumen: sin jerga técnica en pregunta, hallazgo, significado y recomendación",
         lambda: all(len(str(resumen[k]).strip()) >= 25 for k in ["pregunta", "hallazgo", "significado", "recomendacion"]) and _sin_jerga()),
        ("hallazgo: una cifra y su comparación",
         lambda: re.search(r"\d", resumen["hallazgo"]) is not None
         and any(c in resumen["hallazgo"].lower() for c in COMPARACION)),
        ("limitación concreta (≥ 40 caracteres, no 'no hay' ni solo 'faltan datos')",
         lambda: len(resumen["limitacion"]) >= 40 and not resumen["limitacion"].lower().startswith("no hay")
         and resumen["limitacion"].lower().strip(" .") not in {"faltan datos", "falta más data", "faltan más datos"}),
        ("frase técnica: cifra ± desviación, comparación y origen",
         lambda: "±" in frase_tecnica and len(re.findall(r"\d+,\d+", frase_tecnica)) >= 3
         and any(o in frase_tecnica.lower() for o in ["celda", ".ipynb", "resultados_cv", "resultado_test"])),
        ("conclusión del ejercicio", lambda: conclusion_ej1 == "b"),
    ])


verificar_ej1()

---
## Ejercicio 2 — Arreglar tres figuras engañosas (≈25 min)

Una figura buena se entiende **sin leer el texto de al lado**. El checklist: **título = mensaje** (dice la conclusión) · una idea por figura · destacar lo importante y dejar el resto en gris · etiquetas legibles · **fuente y *n* al pie** · la incertidumbre y la comparación a la vista.

Corrijan primero lo que **engaña**, después lo que **estorba** y al final lo estético. Ver `lab14-guia-comunicar-resultados.md`, §3 y §4.

### 2.1 Primero, el error
Las tres figuras del borrador. Mírenlas y anoten qué conclusión sacaría un lector apurado de cada una.

In [ ]:
FRANJAS = pd.cut(df["hora_pedido"], [9, 12, 15, 19, 23], labels=["10–12 h", "13–15 h", "16–19 h", "20–23 h"])
tasa_canal = df.groupby("canal")["retraso"].mean().reindex(["app", "web", "telefono"])
n_canal = df["canal"].value_counts().reindex(tasa_canal.index)
tasa_franja_zona = df.groupby([FRANJAS, "zona"], observed=True)["retraso"].mean().unstack()
resumen_cv = resultados_cv.groupby("modelo", sort=False)[["costo_por_pedido", "sensibilidad", "pr_auc"]].mean()

# Figura A del borrador
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(tasa_canal.index, tasa_canal.values, color=["tab:red", "tab:blue", "tab:green"])
ax.set_ylim(0.108, 0.117)
ax.set_title("Tasa de retraso por canal")
plt.show()

# Figura B del borrador
fig, ax = plt.subplots(figsize=(6, 3.5))
tasa_franja_zona.plot(ax=ax, marker="o")
ax.set_title("Tasa de retraso por zona y franja horaria")
ax.legend(fontsize=11, ncol=2)
plt.show()

# Figura C del borrador
fig, ax = plt.subplots(figsize=(7, 3.5))
resumen_cv.plot.bar(ax=ax)
ax.set_title("Resultados")
plt.show()

*En la A, ¿la app tiene "el doble" de retrasos que el teléfono? En la B, ¿qué hay que mirar? En la C, ¿cuál modelo conviene, con qué incertidumbre y frente a qué?*

### 2.2 ✏️ TU TURNO
Rehagan las tres con Matplotlib, usando `tasa_canal`, `n_canal`, `tasa_franja_zona` y `resultados_cv`, y guárdenlas con `guardar_figura`:

1. `fig_a` (barras de `tasa_canal`): **eje Y desde 0**, título que diga la conclusión (por ejemplo, si la diferencia entre canales importa o no), y una nota al pie con la fuente y el ***n*** (`fig.text(..., "Fuente: … · n = …")`). Guardar como `lab14_fig_a.png`.
2. `fig_b` (`tasa_franja_zona`, una línea por zona): **una sola zona destacada en color** y las otras cinco **en gris**, título-mensaje sobre esa zona, etiqueta directa en la línea destacada en vez de una leyenda grande (o sin leyenda). Guardar como `lab14_fig_b.png`.
3. `fig_c` (comparación de opciones): **una sola métrica**, el costo por pedido, con la **media y la desviación** entre folds como barra de error (`ax.barh(..., xerr=...)` o `ax.errorbar`), una **línea de referencia** para la mejor política trivial (`ax.axvline` o `ax.axhline`), título-mensaje y nota con el *n*. Guardar como `lab14_fig_c.png`.

In [ ]:
# ✏️ TU TURNO — Figura A
fig_a, ax = plt.subplots(figsize=(5, 3.5))
# ...
# guardar_figura(fig_a, "lab14_fig_a.png")

In [ ]:
# ✏️ TU TURNO — Figura B
fig_b, ax = plt.subplots(figsize=(6, 3.5))
# ...
# guardar_figura(fig_b, "lab14_fig_b.png")

In [ ]:
# ✏️ TU TURNO — Figura C
costo_cv = resultados_cv.groupby("modelo")["costo_por_pedido"].agg(["mean", "std"]).sort_values("mean")
fig_c, ax = plt.subplots(figsize=(6, 3.5))
# ...
# guardar_figura(fig_c, "lab14_fig_c.png")

### ✅ Verificación automática

In [ ]:
def _lineas_datos(ax):
    return [l for l in ax.get_lines() if len(l.get_xdata()) > 2]


def _hay_referencia(ax):
    for l in ax.get_lines():
        xd, yd = np.asarray(l.get_xdata(), dtype=float), np.asarray(l.get_ydata(), dtype=float)
        if len(xd) == 2 and (np.allclose(xd[0], xd[1]) or np.allclose(yd[0], yd[1])):
            return True
    return False


def _hay_barras_error(ax):
    return any(isinstance(c, ErrorbarContainer) for c in ax.containers) or \
        any(getattr(c, "errorbar", None) is not None for c in ax.containers)


def _leyenda_chica(ax):
    leg = ax.get_legend()
    return leg is None or len(leg.get_texts()) <= 2


def _barras(ax):
    return [p for p in ax.patches if p.get_width() > 0 and p.get_height() > 0]


def verificar_ej2():
    ax_a = fig_a.axes[0]
    ax_b = fig_b.axes[0]
    ax_c = fig_c.axes[0]
    return ejecutar_checks([
        ("A: las tres barras con el eje Y desde 0", lambda: len(_barras(ax_a)) == 3 and abs(ax_a.get_ylim()[0]) < 1e-9),
        ("A: título-mensaje (no genérico, ≥ 25 caracteres)", lambda: titulo_es_mensaje(ax_a)),
        ("A: nota con fuente y n", lambda: tiene_nota_n(fig_a)),
        ("A: guardada en figs/lab14_fig_a.png", lambda: (FIGS / "lab14_fig_a.png").exists()),
        ("B: las seis zonas, una sola en color y cinco en gris",
         lambda: len(_lineas_datos(ax_b)) >= 6 and sum(not es_gris(l.get_color()) for l in _lineas_datos(ax_b)) == 1),
        ("B: título-mensaje y sin leyenda grande", lambda: titulo_es_mensaje(ax_b) and _leyenda_chica(ax_b)),
        ("B: guardada en figs/lab14_fig_b.png", lambda: (FIGS / "lab14_fig_b.png").exists()),
        ("C: una sola métrica (entre 3 y 5 barras)", lambda: 3 <= len(_barras(ax_c)) <= 5),
        ("C: barras de error (desviación entre folds)", lambda: _hay_barras_error(ax_c)),
        ("C: línea de referencia de la política trivial", lambda: _hay_referencia(ax_c)),
        ("C: título-mensaje y nota con n", lambda: titulo_es_mensaje(ax_c) and tiene_nota_n(fig_c)),
        ("C: guardada en figs/lab14_fig_c.png", lambda: (FIGS / "lab14_fig_c.png").exists()),
    ])


verificar_ej2()

---
## Ejercicio 3 — Tabla de resultados lista para el informe (≈25 min)

Una tabla bien hecha **se copia al informe sin editarla a mano**: si hay que corregirla, el proceso no es reproducible. ¿Tabla o gráfico? **Tabla** si el lector necesita valores exactos; **gráfico** si necesita ver un patrón (la Figura C ya muestra el patrón; la tabla da los números).

Ver `lab14-guia-comunicar-resultados.md`, §5.

### 3.1 Primero, el error
La "tabla" del borrador, copiada de la consola.

In [ ]:
print(resultados_cv.groupby("modelo")[["costo_por_pedido", "sensibilidad", "pr_auc"]].mean())

*Orden alfabético, seis decimales con punto, sin desviación, las políticas triviales mezcladas con los modelos y sin decir qué es mejor (¿mayor o menor costo?).*

### 3.2 ✏️ TU TURNO
1. `tabla_resultados` = DataFrame con **una fila por opción** (las cinco: dos políticas triviales y tres modelos), índice con el nombre de la opción (`index.name = "Opción"`) y **exactamente** las columnas `"Costo por pedido"`, `"Sensibilidad"` y `"PR-AUC"`. Cada celda es texto con coma decimal y tres decimales, **media ± desviación** entre folds (`"0,616 ± 0,078"`; usen `fmt`). Filas **ordenadas por costo medio, de menor a mayor**, y la celda de costo de la mejor opción **en negrita** (`"**0,616 ± 0,078**"`).
2. Exportarla con `df_a_markdown(tabla_resultados, REPORTS / "tabla_resultados.md", nota=...)`, con una nota que diga qué es cada número (media ± desviación, cuántos folds y pedidos, umbral, qué significa "mejor").
3. `veredicto` = `"ganador"` o `"empate"` entre las dos mejores opciones (regla del Lab 11: si la diferencia de medias es menor que la desviación, es **empate práctico**), `elegido` = el modelo que recomiendan y `frase_eleccion` = una frase que lo justifique como se escribiría en el informe.
4. `conclusion_ej3` = `"a"`, `"b"` o `"c"`:
   - a) El bosque aleatorio es peor que la logística y hay que descartarlo en el informe.
   - b) La diferencia entre logística y bosque es menor que su desviación: se informa como empate práctico y se elige el más simple de explicar, declarándolo.
   - c) La logística es claramente superior: tiene el menor costo medio, así que no hace falta mostrar la desviación.

In [ ]:
# ✏️ TU TURNO
agg = resultados_cv.groupby("modelo")[["costo_por_pedido", "sensibilidad", "pr_auc"]].agg(["mean", "std"])
# agg["costo_por_pedido"]["mean"], agg["costo_por_pedido"]["std"], ...

tabla_resultados = ...

# df_a_markdown(tabla_resultados, REPORTS / "tabla_resultados.md", nota="...")

veredicto = ""
elegido = ""
frase_eleccion = ""
conclusion_ej3 = ""

### ✅ Verificación automática

In [ ]:
PATRON_CELDA = re.compile(r"^(\*\*)?\d,\d{3} ± \d,\d{3}(\*\*)?$")


def verificar_ej3():
    agg_ = resultados_cv.groupby("modelo")["costo_por_pedido"].mean().sort_values()
    return ejecutar_checks([
        ("tabla: cinco opciones y las tres columnas pedidas",
         lambda: len(tabla_resultados) == 5
         and list(tabla_resultados.columns) == ["Costo por pedido", "Sensibilidad", "PR-AUC"]),
        ("tabla: cada celda es 'media ± desviación' con coma y tres decimales",
         lambda: all(PATRON_CELDA.match(str(v)) for v in tabla_resultados.to_numpy().ravel())),
        ("tabla: ordenada por costo medio, de menor a mayor",
         lambda: list(tabla_resultados.index) == list(agg_.index)),
        ("tabla: la mejor opción en negrita, y solo esa",
         lambda: str(tabla_resultados.iloc[0, 0]).startswith("**")
         and sum(str(v).startswith("**") for v in tabla_resultados.to_numpy().ravel()) == 1),
        ("tabla: valores correctos (costo de la mejor opción)",
         lambda: str(tabla_resultados.iloc[0, 0]).strip("*").startswith(fmt(agg_.iloc[0]))),
        ("archivo reports/tabla_resultados.md con tabla y nota",
         lambda: (REPORTS / "tabla_resultados.md").exists()
         and all(s in (REPORTS / "tabla_resultados.md").read_text(encoding="utf-8") for s in ["|", "±", "fold"])),
        ("veredicto y modelo elegido", lambda: veredicto == "empate" and elegido == "Regresión logística"),
        ("frase de elección con cifras", lambda: len(frase_eleccion) >= 60 and len(re.findall(r"\d+,\d+", frase_eleccion)) >= 2),
        ("conclusión del ejercicio", lambda: conclusion_ej3 == "b"),
    ])


verificar_ej3()

### 3.3 Extra (opcional) · ✏️ Una página de resumen

Para quien tiene un minuto: **una pantalla** con el mensaje, tres cifras clave y una figura principal.

- **Parte A (todos):** una figura de Matplotlib con el título-mensaje arriba, un recuadro de texto con **tres cifras clave** (cada una con su comparación) y la Figura C (o una versión simplificada) abajo. Guardarla como `reports/pagina_resumen.png`.
- **Parte B (opcional, solo si hay `plotly`):** la misma página como **HTML autocontenido** con un filtro (por ejemplo, un menú por zona con `px.bar(...)`), guardada con `fig.write_html(REPORTS / "pagina_resumen.html", include_plotlyjs=True)`. Debe abrirse con doble clic, sin internet.

Recuerden: **el dashboard es opcional**. Una buena página de resumen vale lo mismo para este curso. Ver `lab14-guia-comunicar-resultados.md`, §6 y §7.

In [ ]:
# ✏️ TU TURNO (opcional) — Parte A
# fig_pag = plt.figure(figsize=(8, 6))
# ...
# REPORTS.mkdir(exist_ok=True); fig_pag.savefig(REPORTS / "pagina_resumen.png", dpi=150, bbox_inches="tight")

# Parte B (solo si HAY_PLOTLY)
# if HAY_PLOTLY:
#     ...

In [ ]:
def verificar_ej3_extra():
    png = REPORTS / "pagina_resumen.png"
    html = REPORTS / "pagina_resumen.html"
    if not png.exists():
        print("⊘ extra no realizado (no cuenta como falla)")
        return True
    print("✓ Parte A: reports/pagina_resumen.png")
    if not HAY_PLOTLY:
        print("⊘ Parte B omitida: plotly no está instalado")
    elif html.exists() and html.stat().st_size > 1_000_000:
        print("✓ Parte B: reports/pagina_resumen.html autocontenido (abre sin internet)")
    elif html.exists():
        print("✗ Parte B: el HTML no es autocontenido (usen include_plotlyjs=True)")
    else:
        print("⊘ Parte B no realizada")
    return True


verificar_ej3_extra()

---
## Proyecto — Resumen ejecutivo y *pitch* de 3 minutos (S26–S27, ≈40 min)

Sin celdas de código obligatorias. Copien estas plantillas al documento del Informe 5 o a un archivo `reports/` de su repositorio.

**1. Resumen ejecutivo de su proyecto (10 min, cinco líneas, estructura del Ejercicio 1):**

| Parte | Su texto (una o dos frases) | ¿De qué celda sale la cifra? |
|---|---|---|
| Pregunta | | — |
| Hallazgo (una cifra + su comparación, traducida) | | |
| Qué significa | | |
| Qué se recomienda | | — |
| Qué limita la conclusión | | |

**2. Guion del *pitch* no técnico de 3 minutos (lo dicen en voz alta en el bloque siguiente, 30 min en total: 3 min por equipo + preguntas + retroalimentación):**

| Tramo | Tiempo | Qué dicen | Qué muestran |
|---|---|---|---|
| La pregunta y por qué importa | 0:00–0:30 | | — |
| El hallazgo, con la figura principal | 0:30–2:00 | | su figura principal, con título-mensaje |
| Qué significa y qué recomiendan | 2:00–2:30 | | — |
| La limitación más seria | 2:30–3:00 | | — |

Reglas: **ambos integrantes hablan**, nada de nombres de algoritmos ni métricas sin traducir, una sola cifra para recordar.

**3. Hoja de retroalimentación (la llena el público para cada equipo):**

| Lo que quedó claro | Lo que se confundió | Una pregunta que me quedó |
|---|---|---|
| | | |

Escala de 1 a 5: claridad para no técnicos · cifra principal comprensible · uso del tiempo · figura. *Críticas al trabajo, nunca a las personas.*

> **Para la próxima clase (jueves 19-nov): ensayo cronometrado de la exposición.** Traer un **guion borrador de 12 minutos** (primero la versión para audiencia no técnica, después la técnica) y las diapositivas en el estado en que estén. **Informe 5 y exposición final: jueves 03-dic.**